# European Satellite Land-Use Patch Classification

A **remote-sensing research team** needs to classify land-use patterns from satellite imagery captured over Europe. Each image is a 64x64 overhead patch belonging to one of ten land-use categories.

## Tasks

1. Load the training images using `torchvision.datasets.ImageFolder` with a basic `ToTensor()` transform and verify that raw tensor values fall in `[0.0, 1.0]`.

2. Build a **training transform pipeline** named `train_transform` as a `transforms.Compose` that includes (in order): `Resize`, `RandomHorizontalFlip`, `RandomVerticalFlip`, `ColorJitter`, `ToTensor`, and `Normalize` using EuroSAT RGB means `[0.3444, 0.3803, 0.4078]` and stds `[0.2038, 0.1367, 0.1148]`.

3. Build a **deterministic evaluation transform pipeline** named `eval_transform` as a `transforms.Compose` that includes only `Resize`, `ToTensor`, and `Normalize` (same statistics). No stochastic augmentations.

4. Verify that applying `eval_transform` to a single training image produces a tensor with shape `(3, 64, 64)` and dtype `torch.float32`.

5. Train a small CNN (`model`) on the training images and evaluate its accuracy.

**Required variable names:** `train_transform`, `eval_transform`, `model`

In [ ]:
# Run this cell before writing your solution
import warnings
warnings.filterwarnings('ignore')

import os
import zipfile
import tempfile

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

# Extract the training archive into the images/ folder
zipfile.ZipFile("images_CV3.zip").extractall("images")

print("Archive extracted.")
print("Classes found:", sorted(os.listdir("images/train")))

In [ ]:
# Write your code here

torch.manual_seed(42)

# EuroSAT RGB channel statistics
EUROSAT_MEAN = [0.3444, 0.3803, 0.4078]
EUROSAT_STD  = [0.2038, 0.1367, 0.1148]

# -----------------------------------------------------------------
# Task 1 -- Load with basic ToTensor and verify [0, 1] range
# -----------------------------------------------------------------
basic_transform = transforms.Compose([transforms.ToTensor()])
basic_dataset   = ImageFolder("images/train", transform=basic_transform)
sample_img, _   = basic_dataset[0]
print("Raw tensor min:", sample_img.min().item(),
      " max:", sample_img.max().item())
assert sample_img.min() >= 0.0 and sample_img.max() <= 1.0, \
    "Pixel values must be in [0.0, 1.0] after ToTensor"

# -----------------------------------------------------------------
# Task 2 -- Training transform (with stochastic augmentations)
# RandomVerticalFlip is appropriate for overhead satellite imagery
# because there is no canonical 'up' direction in aerial views.
# ColorJitter compensates for seasonal and lighting variation.
# Normalize centres the distribution for faster gradient descent.
# -----------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
])

# -----------------------------------------------------------------
# Task 3 -- Deterministic evaluation transform (no stochastic ops)
# Stochastic augmentations must be excluded at evaluation time so
# that predictions are reproducible and comparable across runs.
# -----------------------------------------------------------------
eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
])

# -----------------------------------------------------------------
# Task 4 -- Shape and dtype verification
# -----------------------------------------------------------------
eval_dataset = ImageFolder("images/train", transform=eval_transform)
t, _         = eval_dataset[0]
print("eval_transform tensor shape:", t.shape, "dtype:", t.dtype)
assert t.shape == torch.Size([3, 64, 64]), "Expected shape (3, 64, 64)"
assert t.dtype == torch.float32,          "Expected dtype torch.float32"

# -----------------------------------------------------------------
# Task 5 -- Train a small CNN
# -----------------------------------------------------------------
train_dataset = ImageFolder("images/train", transform=train_transform)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model     = SmallCNN(num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 15
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for imgs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}  loss: {running_loss/len(train_loader):.4f}")

print("Training complete. Model is ready for evaluation.")